In [ ]:
# ==============================================================================
# PIPELINE DEPENDENCY & IMPACT TRACKING
# ==============================================================================
# Upstream Dependencies:
#   - [Notebook]: notebooks/01_bronze_ingestion.py
#   - [Notebook]: notebooks/02_silver_transformation.ipynb
#   - [Notebook]: notebooks/03_gold_dimensional_modeling.ipynb
#   - [Module]:   notebooks/00_config.py
#   - [Module]:   notebooks/00_audit_logger.py
# Downstream Impact:
#   - [Executes]: Full Medallion Pipeline (Raw -> Bronze -> Silver -> Gold)
#   - [Updates]:  Delta tables across all layers and pipeline_execution_logs
# Risk & Side Effects:
#   - Triggers cascading execution; ensures idempotent processing via lookback and Delta MERGE
# ==============================================================================

"""
Project Panopticon: Master Pipeline Orchestrator
Executes Bronze Ingestion -> Silver Transformation -> Gold Dimensional Modeling sequentially.
Can be run interactively in Databricks or scheduled as a single-task Databricks Job.
"""

from datetime import datetime, timezone, timedelta

# Define widgets for pipeline parameterization
yesterday = (datetime.now(timezone.utc).date() - timedelta(days=1)).isoformat()
dbutils.widgets.text("batch_date", yesterday, "1. Batch Date (YYYY-MM-DD)")
dbutils.widgets.dropdown("load_type", "incremental", ["full", "incremental", "backfill"], "2. Load Type")
dbutils.widgets.dropdown("country_code", "PK", ["PK", "IR", "RU", "ALL"], "3. Country Code")
dbutils.widgets.text("max_records", "1000", "4. Max Records (Bronze)")
dbutils.widgets.text("lookback_days", "0", "5. Lookback Days")


In [ ]:
# Resolve runtime parameters
batch_date = dbutils.widgets.get("batch_date").strip()
load_type = dbutils.widgets.get("load_type").strip().lower()
country_code = dbutils.widgets.get("country_code").strip().upper()
max_records = dbutils.widgets.get("max_records").strip()
lookback_days = dbutils.widgets.get("lookback_days").strip()

print("=" * 75)
print(" PROJECT PANOPTICON: MASTER END-TO-END PIPELINE ORCHESTRATOR")
print("=" * 75)
print(f" Target Date   : {batch_date}")
print(f" Load Type     : {load_type}")
print(f" Country Target: {country_code}")
print(f" Max Records   : {max_records}")
print(f" Lookback Days : {lookback_days}")
print("=" * 75)


In [ ]:
# Stage 1: Bronze Ingestion (OONI API -> Bronze Delta Table)
print("\n[1/3] Launching Stage 1: Bronze Ingestion...")
start_bronze = datetime.now(timezone.utc)

bronze_params = {
    "batch_date": batch_date,
    "load_type": load_type,
    "country_code": country_code if country_code != "ALL" else "PK",
    "max_records": max_records,
    "lookback_days": lookback_days
}

bronze_res = dbutils.notebook.run("./01_bronze_ingestion", timeout_seconds=1800, arguments=bronze_params)
dur_bronze = (datetime.now(timezone.utc) - start_bronze).total_seconds()
print(f"[1/3] Bronze Ingestion completed in {dur_bronze:.1f}s")


In [ ]:
# Stage 2: Silver Transformation (Bronze -> Silver Cleaning, Masking, Enrichment)
print("\n[2/3] Launching Stage 2: Silver Transformation...")
start_silver = datetime.now(timezone.utc)

silver_params = {
    "batch_date": batch_date,
    "load_type": load_type,
    "country_filter": country_code,
    "lookback_days": lookback_days
}

silver_res = dbutils.notebook.run("./02_silver_transformation", timeout_seconds=1800, arguments=silver_params)
dur_silver = (datetime.now(timezone.utc) - start_silver).total_seconds()
print(f"[2/3] Silver Transformation completed in {dur_silver:.1f}s")


In [ ]:
# Stage 3: Gold Dimensional Modeling (Silver -> Gold Star Schema & Marts)
print("\n[3/3] Launching Stage 3: Gold Dimensional Modeling...")
start_gold = datetime.now(timezone.utc)

gold_params = {
    "batch_date": batch_date,
    "load_type": load_type,
    "country_filter": country_code,
    "lookback_days": lookback_days
}

gold_res = dbutils.notebook.run("./03_gold_dimensional_modeling", timeout_seconds=1800, arguments=gold_params)
dur_gold = (datetime.now(timezone.utc) - start_gold).total_seconds()
print(f"[3/3] Gold Dimensional Modeling completed in {dur_gold:.1f}s")
print("\n" + "=" * 75)
print(" ALL 3 PIPELINE TIERS EXECUTED SUCCESSFULLY!")
print("=" * 75)


In [ ]:
# Stage 4: Verification & Audit Inspection
# Query the centralized audit logs to review end-to-end metrics
display(
    spark.sql("""
        SELECT log_id, pipeline_layer, batch_parameter, load_type, start_time, end_time,
               duration_seconds, rows_read, rows_inserted, rows_updated, rows_quarantined, status
        FROM delta.`/Volumes/workspace/default/panopticon/logs/pipeline_execution_logs`
        ORDER BY start_time DESC
        LIMIT 10
    """)
)
